# Sync Gold Tables to Lakebase

Creates synced tables to replicate gold layer data into Lakebase for low-latency app access.

**Prerequisites:**
- Gold tables exist with CDF enabled (pipeline must have run)
- Lakebase project `db-residential-copilot` is ONLINE (notebook 02)

**Synced tables created:**
- `app.portfolio_metrics` ← `db_residential_demo.gold.gold_portfolio_property_metrics`
- `app.portfolio_time_series` ← `db_residential_demo.gold.gold_portfolio_time_series`

In [ ]:
%pip install -U "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import (
    SyncedDatabaseTable,
    SyncedTableSpec,
    NewPipelineSpec,
    SyncedTableSchedulingPolicy,
)

w = WorkspaceClient()

PROJECT_ID = "db-residential-copilot"
CATALOG = "db_residential_demo"
LAKEBASE_CATALOG = f"db-residential-copilot.production.databricks_postgres"

## Verify CDF is Enabled on Gold Tables

In [ ]:
gold_tables = [
    f"{CATALOG}.gold.gold_portfolio_property_metrics",
    f"{CATALOG}.gold.gold_portfolio_time_series",
]

for table in gold_tables:
    props = spark.sql(f"SHOW TBLPROPERTIES {table}").filter("key = 'delta.enableChangeDataFeed'")
    cdf_enabled = props.count() > 0 and props.first()["value"] == "true"
    status = "CDF enabled" if cdf_enabled else "CDF NOT enabled — sync may use snapshot mode"
    print(f"  {table}: {status}")

## Create Synced Tables

In [ ]:
synced_table_configs = [
    {
        "source": f"{CATALOG}.gold.gold_portfolio_property_metrics",
        "target": f"{LAKEBASE_CATALOG}.app.portfolio_metrics",
        "pk": ["property_id"],
    },
    {
        "source": f"{CATALOG}.gold.gold_portfolio_time_series",
        "target": f"{LAKEBASE_CATALOG}.app.portfolio_time_series",
        "pk": ["rent_month"],
    },
]

for config in synced_table_configs:
    target_name = config["target"]
    print(f"\nCreating synced table: {target_name}")
    print(f"  Source: {config['source']}")
    print(f"  Primary key: {config['pk']}")

    try:
        # Check if it already exists
        existing = w.database.get_synced_database_table(name=target_name)
        print(f"  Already exists (state: {existing.data_synchronization_status.detailed_state})")
    except Exception:
        # Create the synced table
        w.database.create_synced_database_table(
            SyncedDatabaseTable(
                name=target_name,
                spec=SyncedTableSpec(
                    source_table_full_name=config["source"],
                    primary_key_columns=config["pk"],
                    scheduling_policy=SyncedTableSchedulingPolicy.TRIGGERED,
                    new_pipeline_spec=NewPipelineSpec(
                        storage_catalog=CATALOG,
                        storage_schema="gold"
                    )
                ),
            )
        )
        print(f"  Created successfully.")

## Check Sync Status

In [ ]:
import time

print("Waiting for sync to complete...")
for config in synced_table_configs:
    target_name = config["target"]
    for attempt in range(30):
        status = w.database.get_synced_database_table(name=target_name)
        state = status.data_synchronization_status.detailed_state
        if "ONLINE" in str(state) or "ACTIVE" in str(state):
            print(f"  {target_name}: {state}")
            break
        elif "FAIL" in str(state) or "ERROR" in str(state):
            print(f"  {target_name}: FAILED — {status.data_synchronization_status.message}")
            break
        else:
            if attempt % 5 == 0:
                print(f"  {target_name}: {state} (waiting...)")
            time.sleep(10)
    else:
        print(f"  {target_name}: timed out — check status manually")

print("\nSync setup complete.")